# OCSD – Chạy toàn bộ thực nghiệm (Run all)

Khóa luận: *Sinh ảnh cảnh nhất quán đối tượng từ phác thảo và văn bản bằng mô hình khuếch tán* – Lê Hoàng Bảo (21090331), Thái Bá Cường (21050681); GVHD: TS. Nguyễn Thanh Chuyên.

**Cập nhật 03/11/2026:** đã nhận đủ `foreground_screen` của run `20261103T035056Z_dc40e4b5`. Đã nhận review: A–D đủ số lượng/hình dáng; A/B chính xác nhất nhưng chưa tự nhiên. **Không cần chấm lại hay chạy lại phép thử foreground.** [Xem A–D bằng PNG, không cần HTML](https://raw.githubusercontent.com/bobbibao/ocsd-thesis/main/docs/reviews/foreground_20261103_ABCD.png). Hướng mới: điều kiện hóa trực tiếp từng đối tượng bằng nét phác thảo với lõi InstanceDiffusion chính thức. Đây là ứng viên đang kiểm chứng, chưa có kết quả chứng minh chất lượng ảnh tốt hơn.

**Update 2026-11-04:** the canary and the 8-scene development study are done and reviewed; do not rerun them ([dev results](https://github.com/bobbibao/ocsd-thesis/blob/main/docs/DIRECT_DEV_20261104_RESULTS.md)). Held-out v1, the silhouette dev study and held-out v2 are done; do not rerun them ([held-out v2 results](https://github.com/bobbibao/ocsd-thesis/blob/main/docs/HELDOUT2_20261104_RESULTS.md)).

The naturalness dev study (dev v3) is done; do not rerun it ([dev v3 results](https://github.com/bobbibao/ocsd-thesis/blob/main/docs/DIRECT_DEV3_20261104_RESULTS.md)).

Held-out v3 is done: OCSD v3 beats the original OCSD; do not rerun it ([held-out v3 results](https://github.com/bobbibao/ocsd-thesis/blob/main/docs/HELDOUT3_20261105_RESULTS.md)).

The paper3 confirmation of OCSD v3 is done; do not rerun it ([results](https://github.com/bobbibao/ocsd-thesis/blob/main/docs/PAPER3_20261105_RESULTS.md)). OCSD v3 replaces M2-M5 with an external generator, so it is kept only as a comparison.

**Next step (2026-11-13): OCSD-XL development round 2.** Round 1 showed that the object step (M2) draws white cut-outs on SDXL; round 2 fixes M2 and repeats the study on 9 other development scenes. Runtime → Change runtime type → **A100 GPU** (High-RAM) → keep `RUN_MODE="xl_dev2"` → **Run all**. 9 scenes × 2 seeds × 13 arms = 234 images, about 2 A100 hours. If Colab stops, Run all again: finished images are kept. Two ZIP files download at the end (also on Drive in `xl_dev_v2/results/`); send `REVIEW_xl_dev_v2.zip` to Claude. [Round 1 results](https://github.com/bobbibao/ocsd-thesis/blob/claude/project-thread-ezrky1/docs/XL_DEV1_RESULTS.md), [Protocol](https://github.com/bobbibao/ocsd-thesis/blob/claude/project-thread-ezrky1/docs/XL_DEV_PROTOCOL.md).

Chỉ khi chủ động tái lập thí nghiệm cũ: chọn `RUN_MODE="experiment"` và GPU L4/A100. Luồng này chạy 4 giai đoạn: (A) dựng bộ dữ liệu, (B) sinh ảnh cho E1–E4 và khảo sát α, (C) đánh giá, (D) xuất bảng/hình/tóm tắt. Mọi kết quả nằm trong `DRIVE_ROOT/results/runs/<run_id>/` (`summary.md`, `summary.json`, `tables/`, `figures/`, `progress.json`). Lưu RUN_ID để resume. Chỉ tái sử dụng kết quả cùng code, cấu hình, thư viện và dữ liệu. Xem docs/EXPERIMENT_RUNBOOK.md.

In [1]:
#@title 1. Cấu hình (chỉ sửa ô này)
RUN_MODE   = "xl_dev2"  #@param ["xl_dev2", "xl_dev", "paper3", "heldout3", "naturalness_dev", "heldout2", "silhouette_dev", "heldout", "direct_dev", "direct_canary", "direct_preflight", "experiment"]
REPO_URL   = "https://github.com/bobbibao/ocsd-thesis.git"  #@param {type:"string"}
BRANCH     = "claude/project-thread-ezrky1"                                          #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"                 #@param {type:"string"}
TIER       = "foreground_screen"   #@param ["foreground_screen", "clean_screen", "count_screen", "instance_screen", "semantic_screen", "repair_screen", "screen", "pilot", "paper", "full"]
BACKBONE   = "sd15"    #@param ["sd15", "sd21"]
MAX_MINUTES = 20      #@param {type:"number"}
SKIP_JOBS  = ""        #@param {type:"string"}
XL_MAX_MINUTES = 330   #@param {type:"number"}
RUN_ID     = ""        #@param {type:"string"}
PUSH_RESULTS = True    #@param {type:"boolean"}
# pilot: kiểm tra nhanh (~1-2 giờ GPU); paper: đủ số liệu cho khóa luận; full: cho bài báo (cần thêm GPU).
# Có thể bấm "Run all" nhiều lần: ảnh/độ đo đã có trên Drive sẽ được bỏ qua và chạy tiếp phần còn lại.
# SKIP_JOBS: comma-separated job prefixes to leave out, e.g. "pareto" or "pareto,E3_quickdraw_power" (GPU budget).

In [2]:
#@title 2. Gắn Drive, kiểm tra loại runtime, lấy mã nguồn mới
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
import shutil
if RUN_MODE not in ("xl_dev2", "xl_dev", "direct_preflight", "direct_canary", "direct_dev", "silhouette_dev", "naturalness_dev", "heldout", "heldout2", "heldout3", "paper3", "experiment"):
    raise ValueError("Unknown RUN_MODE")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if RUN_MODE == "direct_preflight":
    if RUN_ID:
        raise ValueError("CPU preflight uses a separate log: leave RUN_ID empty")
    if gpu is not None and gpu.returncode == 0:
        raise RuntimeError("Bước này dùng CPU: Runtime > Change runtime type > Hardware accelerator: None, rồi Run all")
    print("CPU preflight: kiểm tra trọng số thật; không sinh ảnh, không cần GH_TOKEN.")
else:
    if RUN_MODE in ("direct_canary", "direct_dev", "silhouette_dev", "naturalness_dev") and RUN_ID:
        raise ValueError("Direct runs resume compatible images automatically: leave RUN_ID empty")
    if gpu is None or gpu.returncode:
        raise RuntimeError("Select an L4/A100 GPU runtime for this mode")
    print(gpu.stdout)
CODE = "/content/KLTN_OCSD"
if not REPO_URL:
    # chưa có repo: dùng bản mã nguồn đã tải lên Drive (thư mục DRIVE_ROOT/code)
    CODE = os.path.join(DRIVE_ROOT, "code")
elif os.path.exists(CODE):
    if not RUN_ID:
        dirty = subprocess.run(["git", "-C", CODE, "status", "--porcelain"], check=True, capture_output=True, text=True).stdout
        if dirty.strip():
            raise RuntimeError("Checkout has local changes. Preserve them; use a fresh runtime for current GitHub code, or set RUN_ID to resume.")
        subprocess.run(["git", "-C", CODE, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", CODE, "merge", "--ff-only", "FETCH_HEAD"], check=True)
    print("Using checkout (explicit RUN_ID preserves its version):", CODE)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, CODE], check=True)
print("Mã nguồn:", CODE, subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
# drop modules imported by an earlier run in this kernel, so the selected code is what runs
for _m in [m for m in sys.modules if m == "ocsd" or m.startswith("ocsd.")]: del sys.modules[_m]
# HF cache on LOCAL disk: Drive cannot store the cache's symlinks (broken files -> "SafetensorError: header too large").
# Models re-download each new session (~2-4 min on Colab).
os.environ["HF_HOME"] = "/content/hf_cache"
# Public models do not require a token; optional HF authentication can use Colab secrets.

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB

Mã nguồn: /content/KLTN_OCSD f52537d843490184c3fc21a147d8af9da40a6957 OCSD-XL round 1 results and round 2: fix the object branch (M2) for SDXL



In [3]:
#@title 3. Kiểm tra mô hình trực tiếp trên CPU và tải ZIP log
if RUN_MODE == "direct_preflight":
    import json, uuid
    from pathlib import Path
    from zipfile import ZipFile, ZIP_DEFLATED
    from google.colab import files
    DIRECT_OUT = Path(DRIVE_ROOT) / "results" / "direct_preflight" / ("attempt_" + uuid.uuid4().hex[:12])
    DIRECT_OUT.mkdir(parents=True)
    setup_log = DIRECT_OUT / "setup.log"
    print("Cài thư viện; dùng lại cache nếu còn. Tải mới: khoảng 13,6 GB checkpoint + 1,71 GB CLIP, chưa gồm tệp tạm.", flush=True)
    installed = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                               os.path.join(CODE, "requirements-direct-preflight.txt")], capture_output=True, text=True)
    setup_log.write_text(installed.stdout + "\n" + installed.stderr)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="", PYTHONUNBUFFERED="1")
    checked = subprocess.run([sys.executable, os.path.join(CODE, "tools", "preflight_direct.py"),
                              "--cache", "/content/ocsd-direct-cache", "--out", str(DIRECT_OUT),
                              "--setup-log", str(setup_log), "--setup-returncode", str(installed.returncode)], env=env)
    latest = DIRECT_OUT / "latest_preflight.json"
    if latest.exists():
        result = json.loads(latest.read_text())
        archive = result["archive"]
        print("Kết quả kiểm tra:", result["status"])
    else:
        # Preserve even an OS kill/import crash; never download an older successful run.
        archive = str(DIRECT_OUT / "ocsd_direct_preflight_incomplete.zip")
        (DIRECT_OUT / "incomplete.json").write_text(json.dumps(dict(status="incomplete", returncode=checked.returncode,
            message="Process stopped before the report completed. Review logs/RAM; no GPU run is recommended."), indent=2))
        with ZipFile(archive, "w", ZIP_DEFLATED) as z:
            for p in DIRECT_OUT.rglob("*"):
                if p.is_file() and p.suffix in (".log", ".json"):
                    z.write(p, str(p.relative_to(DIRECT_OUT)))
        print("Kiểm tra chưa hoàn tất; đã giữ log để tìm lỗi.")
    print("Gửi file ZIP này vào cuộc trò chuyện:", archive)
    print("PASS chỉ xác nhận tương thích CPU. Chưa chứng minh ảnh đẹp hơn; chưa chuyển sang GPU/paper/full.")
    try:
        files.download(archive)
    except Exception as exc:
        print("Không tự tải được; lấy file theo đường dẫn Drive ở trên:", exc)
else:
    print("Bỏ qua CPU preflight trong chế độ này.")

Bỏ qua CPU preflight trong chế độ này.


In [4]:
#@title 3b. Thử GPU tối đa 4 ảnh và tải PNG so sánh + ZIP
if RUN_MODE == "direct_canary":
    import json, uuid
    from pathlib import Path
    from google.colab import files
    CANARY_OUT = Path(DRIVE_ROOT) / "results" / "direct_canary"
    attempt = CANARY_OUT / ("attempt_" + uuid.uuid4().hex[:12]); attempt.mkdir(parents=True)
    print("Canary: 1 cảnh mới × 2 seed × 2 nhánh = 4 ảnh; tối đa 600 giây cho tiến trình sau cài thư viện.", flush=True)
    print("Tải mới khoảng 13,6 GB + 1,71 GB. Chỉ sinh phần chưa hoàn tất; không tự thử lại nếu lỗi.", flush=True)
    installed = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                               os.path.join(CODE, "requirements-direct-preflight.txt")], capture_output=True, text=True)
    (attempt / "setup.log").write_text(installed.stdout + "\n" + installed.stderr)
    script = os.path.join(CODE, "tools", "run_direct_canary.py")
    interrupted = False
    try:
        checked = subprocess.run([sys.executable, script, "--cache", "/content/ocsd-direct-cache",
            "--out", str(CANARY_OUT), "--attempt", str(attempt), "--setup-returncode", str(installed.returncode)],
            env=dict(os.environ, PYTHONUNBUFFERED="1"), timeout=600)
        interrupted = not (attempt / "result.json").exists()
    except subprocess.TimeoutExpired:
        interrupted = True
        print("Đã dừng tại giới hạn 600 giây; giữ lại ảnh đã hoàn tất và log.")
    if interrupted:
        subprocess.run([sys.executable, script, "--attempt", str(attempt), "--finalize-only", "--interrupted"], timeout=60)
    result_path = attempt / "result.json"
    if not result_path.exists():
        raise RuntimeError(f"Không đóng gói được; giữ thư mục này để kiểm tra: {attempt}")
    result = json.loads(result_path.read_text())
    print("Trạng thái:", result["status"], "| ZIP:", result["archive"])
    if result["status"] == "complete":
        from IPython.display import Image, display
        for seed in (0, 1):
            display(Image(filename=str(Path(result["directory"]) / f"SO_SANH_seed{seed}.png"), width=1152))
        print("Đọc run/HUONG_DAN.txt; điền run/REVIEW_TEMPLATE.txt hoặc trả lời trong chat. Complete chưa phải chất lượng đạt.")
    else:
        print("Gửi ZIP để kiểm tra lỗi/ngân sách; chưa tự tăng số ảnh, số bước hoặc chuyển paper/full.")
    print("Lượt GPU đã dừng. Tải ZIP rồi ngắt runtime GPU khi không dùng để tránh thời gian nhàn rỗi.")
    try:
        files.download(result["archive"])
    except Exception as exc:
        print("Lấy ZIP theo đường dẫn Drive ở trên:", exc)
else:
    print("Bỏ qua canary GPU trong chế độ này.")

Bỏ qua canary GPU trong chế độ này.


In [5]:
#@title 3c. Development study (8 dev scenes x 2 seeds), then download ZIPs
if RUN_MODE in ("direct_dev", "silhouette_dev", "naturalness_dev"):
    import json, uuid
    from pathlib import Path
    from google.colab import files
    DEV_VERSION = {"direct_dev": "v1", "silhouette_dev": "v2", "naturalness_dev": "v3"}[RUN_MODE]
    DEV_OUT = Path(DRIVE_ROOT) / "results" / {"v1": "direct_dev", "v2": "direct_dev2", "v3": "direct_dev3"}[DEV_VERSION]
    DEV_SECONDS = 3000 if DEV_VERSION == "v3" else 2400
    attempt = DEV_OUT / ("attempt_" + uuid.uuid4().hex[:12]); attempt.mkdir(parents=True)
    if DEV_VERSION == "v3":
        print("Locked naturalness study: 32 InstanceDiffusion images + 32 SDXL-Refiner images + SAM sketch fidelity,", flush=True)
        print("at most 3000 s. Extra downloads: SDXL refiner ~6 GB, SAM ViT-H ~2.6 GB.", flush=True)
    else:
        print("Locked development study: 48 images, at most 2400 s after library install. Completed images are reused.", flush=True)
    print("May download ~13.6 GB checkpoint + 1.71 GB CLIP if the runtime cache is empty. No automatic retries.", flush=True)
    reqs = ["-r", os.path.join(CODE, "requirements-direct-preflight.txt")]
    if DEV_VERSION == "v3": reqs += ["-r", os.path.join(CODE, "requirements-direct-refiner.txt")]
    installed = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], capture_output=True, text=True)
    (attempt / "setup.log").write_text(installed.stdout + "\n" + installed.stderr)
    script = os.path.join(CODE, "tools", "run_direct_dev.py")
    interrupted = False
    try:
        subprocess.run([sys.executable, script, "--cache", "/content/ocsd-direct-cache",
            "--out", str(DEV_OUT), "--attempt", str(attempt), "--setup-returncode", str(installed.returncode), "--version", DEV_VERSION],
            env=dict(os.environ, PYTHONUNBUFFERED="1"), timeout=DEV_SECONDS)
        interrupted = not (attempt / "result.json").exists()
    except subprocess.TimeoutExpired:
        interrupted = True
        print(f"Stopped at the {DEV_SECONDS} s limit; completed images and logs are kept.")
    if interrupted:
        subprocess.run([sys.executable, script, "--attempt", str(attempt), "--finalize-only", "--interrupted"], timeout=120)
    result_path = attempt / "result.json"
    if not result_path.exists():
        raise RuntimeError(f"Could not package; keep this folder for inspection: {attempt}")
    result = json.loads(result_path.read_text())
    print("Status:", result["status"], "| full ZIP:", result["archive"])
    downloads = [result["archive"]]
    if result["status"] == "complete":
        print("Review ZIP (16 PNG sheets + HOW_TO_REVIEW.txt + REVIEW_TEMPLATE.txt):", result["review_archive"])
        downloads.append(result["review_archive"])
        print("'complete' means all images exist; it is not a quality pass.")
    else:
        print("Send the ZIP so the failure can be checked; do not change settings or rerun with more images.")
    print("GPU work finished. Disconnect the GPU runtime after downloading to avoid idle compute units.")
    for path in downloads:
        try:
            files.download(path)
        except Exception as exc:
            print("Get it from the Drive path above:", path, exc)
else:
    print("Skipping the development study in this mode.")

Skipping the development study in this mode.


In [6]:
#@title 3d. Held-out / paper comparison, part 1: InstanceDiffusion-core arms, then the other methods below
if RUN_MODE in ("heldout", "heldout2", "heldout3", "paper3"):
    import json, uuid
    from pathlib import Path
    HELDOUT_SET = {"heldout": "v1", "heldout2": "v2", "heldout3": "v3", "paper3": "v4"}[RUN_MODE]
    os.environ["OCSD_HELDOUT"] = HELDOUT_SET
    HELDOUT_DIRECT = Path(DRIVE_ROOT) / ("paper_v3" if HELDOUT_SET == "v4" else "heldout_" + HELDOUT_SET) / "direct"
    if HELDOUT_SET == "v4":
        # The 72 paper_v2 evaluation scenes come from the paper benchmark on Drive (checked, then hashed).
        from ocsd import heldout as _h; _h.use("v4")
        _h.import_paper_scenes(os.path.join(DRIVE_ROOT, "data", "benchmarks", "quickdraw"))
    attempt = HELDOUT_DIRECT / ("attempt_" + uuid.uuid4().hex[:12]); attempt.mkdir(parents=True)
    arms = {"v1": "(box, scribble)", "v2": "(box, silhouette)", "v3": "(InstanceDiffusion box, OCSD v3)", "v4": "(InstanceDiffusion box, OCSD v3)"}[HELDOUT_SET]
    n_sc, limit = (72, 14400) if HELDOUT_SET == "v4" else (18, 3600)
    print(f"Locked set {HELDOUT_SET}: {n_sc} scenes x 2 seeds x {arms} = {4*n_sc} images, at most {limit} s. Completed images are reused.", flush=True)
    reqs = ["-r", os.path.join(CODE, "requirements-direct-preflight.txt")]
    if HELDOUT_SET in ("v3", "v4"): reqs += ["-r", os.path.join(CODE, "requirements-direct-refiner.txt")]
    installed = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], capture_output=True, text=True)
    (attempt / "setup.log").write_text(installed.stdout + "\n" + installed.stderr)
    script = os.path.join(CODE, "tools", "run_direct_heldout.py")
    interrupted = False
    try:
        subprocess.run([sys.executable, script, "--cache", "/content/ocsd-direct-cache", "--out", str(HELDOUT_DIRECT),
            "--attempt", str(attempt), "--setup-returncode", str(installed.returncode)],
            env=dict(os.environ, PYTHONUNBUFFERED="1"), timeout=limit)
        interrupted = not (attempt / "result.json").exists()
    except subprocess.TimeoutExpired:
        interrupted = True
    if interrupted:
        subprocess.run([sys.executable, script, "--attempt", str(attempt), "--finalize-only", "--interrupted"], timeout=120)
    result = json.loads((attempt / "result.json").read_text())
    print("Direct arms:", result["status"], "| log ZIP:", result["archive"])
    if result["status"] != "complete":
        try:
            from google.colab import files; files.download(result["archive"])
        except Exception as exc:
            print("Get the log ZIP from Drive:", exc)
        raise RuntimeError("Direct arms incomplete. Run all again to resume (completed images are kept), or send the log ZIP.")
    import gc; gc.collect()
    print("Direct arms complete. Continuing with the other methods (resumable).")
else:
    print("Skipping the held-out comparison in this mode.")

Skipping the held-out comparison in this mode.


In [7]:
#@title 3e. OCSD-XL study (M1-M5 on SDXL, 1024 px): generate, score, package
if RUN_MODE in ("xl_dev", "xl_dev2"):
    XL_PROTOCOL = {"xl_dev": "xl_dev_v1", "xl_dev2": "xl_dev_v2"}[RUN_MODE]
    prop = subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip()
    if int(prop.splitlines()[0]) < 30000:
        raise RuntimeError("OCSD-XL needs about 30 GB of GPU memory: Runtime > Change runtime type > A100 GPU, then Run all")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(CODE, "requirements-colab.txt")], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-2500:])
    if r.returncode != 0:
        raise RuntimeError("pip install failed (see the text above)")
    from ocsd import hfcache, xl_study
    hfcache.repair(os.environ["HF_HOME"])
    print("First run downloads about 20 GB of public models (SDXL, ControlNet, detectors); no token needed.", flush=True)
    print("If Colab stops, just Run all again: finished images are kept.", flush=True)
    XL_OUT = xl_study.main(DRIVE_ROOT, XL_PROTOCOL, max_minutes=XL_MAX_MINUTES)
    from ocsd import sync
    if PUSH_RESULTS:
        sync.try_push_results(os.path.dirname(XL_OUT["summary"]), sync.colab_token())
    from google.colab import files
    for path in (XL_OUT["archive"], XL_OUT["review_archive"]):
        print(path)
        try:
            files.download(path)
        except Exception as exc:
            print("Get it from Drive:", exc)
    print("GPU work finished. Disconnect the GPU runtime after the two ZIP files have downloaded.")
else:
    print("Skipping the OCSD-XL study in this mode.")

 
First run downloads about 20 GB of public models (SDXL, ControlNet, detectors); no token needed.
If Colab stops, just Run all again: finished images are kept.

===== 2026-11-13 09:19:22 xl_dev_v2 =====
[xl] xl_dev_v2: 0/234 images already complete, 234 to generate


tokenizer_config.json:   0%|          | 0.00/737 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

... (75 outputs left out)

/content/drive/MyDrive/KLTN_OCSD/xl_dev_v2/results/REVIEW_xl_dev_v2.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

GPU work finished. Disconnect the GPU runtime after the two ZIP files have downloaded.


In [8]:
#@title 3. Install libraries (keeps Colab's PyTorch), start the Drive run log
if RUN_MODE in ("experiment", "heldout", "heldout2", "heldout3", "paper3"):
    LOGS = os.path.join(DRIVE_ROOT, "results", "logs"); os.makedirs(LOGS, exist_ok=True)
    # Colab ships torchao 0.10, which peft rejects when adding LoRA ("incompatible version of torchao"); OCSD does not use it
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(CODE, "requirements-colab.txt")], capture_output=True, text=True)
    open(os.path.join(LOGS, "pip_install.log"), "w").write(r.stdout + "\n" + r.stderr)
    print(r.stdout[-2000:], r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError(f"pip install failed; see {LOGS}/pip_install.log")
    import importlib, ocsd; importlib.reload(ocsd)
    from ocsd import hfcache
    hfcache.repair(os.environ["HF_HOME"])   # delete broken model files so they download again
    from ocsd import runlog
    from ocsd.config import Paths, ExperimentConfig, TIERS
    EXP_ROOT = DRIVE_ROOT
    if globals().get("RUN_MODE") in ("heldout", "heldout2", "heldout3", "paper3"):
        # Locked held-out comparison: own data root, fixed tier and generation budget
        # (docs/HELDOUT_PROTOCOL.md, HELDOUT2_PROTOCOL.md, HELDOUT3_PROTOCOL.md)
        HELDOUT_SET = {"heldout": "v1", "heldout2": "v2", "heldout3": "v3", "paper3": "v4"}[RUN_MODE]
        os.environ["OCSD_HELDOUT"] = HELDOUT_SET
        from ocsd import heldout as _heldout; _cfg = _heldout.use(HELDOUT_SET)
        EXP_ROOT, TIER = os.path.join(DRIVE_ROOT, "paper_v3" if HELDOUT_SET == "v4" else "heldout_" + HELDOUT_SET), _cfg["tier"]
        MAX_MINUTES = {"v3": 300, "v4": 360}.get(HELDOUT_SET, 180)
    P = Paths(EXP_ROOT); P.makedirs()
    E = ExperimentConfig(backbone=BACKBONE, tier=TIER, skip_jobs=SKIP_JOBS,
                         extra_detectors=() if TIER in ("screen", "repair_screen", "semantic_screen", "instance_screen", "count_screen", "clean_screen", "foreground_screen", "heldout_screen", "heldout_final") else ("owlv2", "owlv2d", "detr"))
    from ocsd.provenance import start_run
    P, RUN = start_run(P, E, run_id=RUN_ID, notebook="notebooks/RUN_ALL.ipynb")
    RUN_ID = P.run_id
    runlog.start(P.results, CODE)
    LOGS = os.path.join(P.results, "logs")
    E.save(os.path.join(P.results, "experiment_config.json"))
    print("RUN_ID:", RUN_ID, "results:", P.results)
    print(E)

## Các ô dưới chỉ chạy khi RUN_MODE=experiment
Phép thử cũ `foreground_screen`: 1 cảnh xe đạp × 4 đối chứng = 4 ảnh cuối. So sánh clean cũ, anchor có nền cảnh, mask học được dạng cứng và dạng mềm. BiRefNet-lite chạy CPU, tải khoảng 224 MB một lần và kiểm tra SHA-256. Không gửi ảnh ra dịch vụ ngoài. Cùng mẫu M2, vị trí, vùng sửa và số bước diffusion; chi phí tách nền ghi riêng. Đã chạy và nhận review: người dùng ưu tiên hai nhánh tách nền, count/shape đạt trên cảnh này, nhưng chưa tự nhiên. Xem docs/FOREGROUND_20261103_RESULTS.md. Không chạy lại sweep này hoặc paper/full theo kết quả hiện tại. Chỉ dùng RUN_ID mới nếu chủ động tái lập phép thử; đây chưa phải model đạt chất lượng.

## A. Bộ dữ liệu
QuickDraw-Scenes (tổng hợp, 12 ô: số đối tượng 1/3/5/8+ × độ phức tạp đơn giản/trung bình/phức tạp) và COCO-Sketch (ảnh thật COCO val2017, phác thảo PiDiNet, mặt nạ thật). Lần đầu mất ~10–20 phút (tải COCO 1 GB).

In [9]:
#@title A. Build benchmarks (skipped if already built for this tier)
if RUN_MODE in ("experiment", "heldout", "heldout2", "heldout3", "paper3"):
    from ocsd import stages
    with RUN.stage("A_data"):
        stages.setup_data(P, TIER, with_coco=TIER not in ("screen", "repair_screen", "semantic_screen", "instance_screen", "count_screen", "clean_screen", "foreground_screen", "heldout_screen", "heldout_final"))
        RUN.seal_data()
        RUN.plan(E)
    from IPython.display import Image, display
    f = os.path.join(P.results, "figures", "benchmark_examples.png")
    if os.path.exists(f): display(Image(f, width=900))

## B. Sinh ảnh
Thứ tự: E3 QuickDraw (mọi baseline + OCSD-lite trên toàn bộ tập) → OCSD và Zhang et al. (cần học định danh, trên tập con phân tầng) → E3 COCO → E4 cắt bỏ → khảo sát α. `MAX_MINUTES` giới hạn thời gian của ô này; chạy lại để tiếp tục.

## T. Tuning
Paper/full tiers only, on the pilot scenes, which are then left out of the evaluation: (1) α × LoRA strength, (2) how M5(b) energy guidance is applied, (3) which of M5(a) / M5(b) to keep, jointly with α. Writes `results/tuning/tuned.json`; finished phases are skipped (phase 3 alone is about 1 A100 hour).

In [10]:
#@title T. Tune OCSD on the pilot scenes (finished phases are skipped)
if RUN_MODE in ("experiment", "heldout", "heldout2", "heldout3", "paper3"):
    import torch
    eng = vis = None
    with RUN.stage("T_tune"):
        eng, vis = stages.tune(P, E, max_minutes=MAX_MINUTES)

In [11]:
#@title B1. Plan and remaining images
if RUN_MODE in ("experiment", "heldout", "heldout2", "heldout3", "paper3"):
    for j in stages.experiment_plan(P, E):
        print(f"{j['name']:22s} {len(j['scenes']):4d} scenes x {len(j['methods']):2d} methods x {len(j['seeds'])} seeds -> {stages._count_todo(P, j)} images left")

In [12]:
#@title B2. Generate images (resumable: finished images are skipped)
if RUN_MODE in ("experiment", "heldout", "heldout2", "heldout3", "paper3"):
    with RUN.stage("B_generate"):
        eng, vis = stages.generate(P, E, max_minutes=MAX_MINUTES, eng=eng, vis=vis)

## C. Đánh giá
OPR, OCE, độ chính xác đếm, mIoU, RA bằng bộ phát hiện **OWLv2** (khác Grounding DINO mà phương pháp dùng bên trong, để tránh tối ưu hóa theo chính độ đo); CLIP score toàn ảnh và cấp đối tượng; ID-Sim (DINOv2); FID/KID; LPIPS.

In [13]:
#@title C. Metrics (only new images are evaluated; FID reused when unchanged)
if RUN_MODE in ("experiment", "heldout", "heldout2", "heldout3", "paper3"):
    if eng is not None:
        del eng
    from ocsd.methods import _GLIGEN
    _GLIGEN.clear()
    import gc
    gc.collect(); torch.cuda.empty_cache()
    with RUN.stage("C_evaluate"):
        vis = stages.evaluate(P, E, vis=vis, fid=TIER not in ("screen", "repair_screen", "semantic_screen", "instance_screen", "count_screen", "clean_screen", "foreground_screen", "heldout_screen", "heldout_final"))

## D. Báo cáo

In [14]:
#@title D. Tables, figures, statistics, summary
if RUN_MODE in ("experiment", "heldout", "heldout2", "heldout3", "paper3"):
    with RUN.stage("D_report"):
        summary = stages.report(P, E)
    RUN.finish(E)  # checksum only after the stage has written its final status
    from IPython.display import Markdown, display
    display(Markdown(open(os.path.join(P.results, "summary.md")).read()))
    for f in summary["figures"]:
        display(Image(f, width=900))

In [15]:
#@title E. Run status (also saved to results/logs/status.json; errors in results/logs/errors/)
if RUN_MODE in ("experiment", "heldout", "heldout2", "heldout3", "paper3"):
    runlog.summary()
    if os.path.exists(os.path.join(LOGS, "LATEST_ERROR.txt")):
        print("\nMost recent error:\n" + open(os.path.join(LOGS, "LATEST_ERROR.txt")).read()[-3000:])

In [16]:
#@title F. Send logs and results to GitHub (branch colab-results) so Claude can read them
if RUN_MODE in ("experiment", "heldout", "heldout2", "heldout3", "paper3"):
    from ocsd import sync
    if PUSH_RESULTS:
        sync.try_push_results(P.results, sync.colab_token())
    else:
        print("Results retained on Drive:", P.results)

In [17]:
#@title G. Held-out mode: download the results ZIP and the blind review ZIP
if RUN_MODE in ("experiment", "heldout", "heldout2", "heldout3", "paper3"):
    if RUN_MODE in ("heldout", "heldout2", "heldout3", "paper3"):
        from google.colab import files
        tag = RUN_MODE
        for name in (f"ocsd_{tag}_results.zip", f"REVIEW_{tag}.zip"):
            path = os.path.join(P.results, name)
            print(path)
            try:
                files.download(path)
            except Exception as exc:
                print("Get it from Drive:", exc)
        print("GPU work finished. Disconnect the GPU runtime after downloading.")